In [8]:
import requests
import pandas as pd
import json

# Configurações da API Local
BASE_URL = "http://localhost:3003"
API_KEY = "UgdRwHpekKRWafd+gA0Q1I72iCeQuKAqArrMEkU43f6UdgOJDsUnqoMKDB+2hhk8st9DNW9gSozSUdvKGI2w89RMMlMGEAEb1hYILnjroouAAMvAvZygGORoaX3DYQi4Dj8KX0mtHRcYRS7XW76BOxpijFVUa3IdhYLhZJIL1uo="
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}
ENDPOINTS = {
    "registration": f"{BASE_URL}/registration",
    "participation": f"{BASE_URL}/business-participation"
}

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [9]:
document_list = [
    "10.526.937/0005-82", "44.069.771/0001-00", "17.159.005/0001-64", 
    "72.908.817/0001-73", "48.066.047/0001-84", "01.488.278/0001-12", 
    "13.241.169/0001-85", "05.885.911/0001-67", "38.518.783/0001-72", 
    "05.444.931/0001-00", "03.532.142/0001-98", "10.794.918/0001-03", 
    "27.819.937/0001-40", "14.085.790/0001-60", "37.569.667/0001-10", 
    "03.277.948/0001-87", "07.839.915/0001-34", "57.506.115/0001-70", 
    "06.094.348/0001-71", "08.695.920/0001-83", "03.094.658/0001-06", 
    "43.823.079/0001-63", "32.698.438/0001-81", "42.004.701/0001-49", 
    "10.355.516/0001-02", "60.806.577/0001-17", "50.228.097/0001-62", 
    "92.672.070/0001-04", "33.544.370/0014-63", "06.325.818/0001-60", 
    "01.498.476/0015-62", "07.317.846/0001-07", "08.415.791/0001-22", 
    "12.591.077/0001-62", "31.093.883/0001-55", "92.751.213/0001-73", 
    "33.054.883/0001-71", "02.394.107/0001-97", "40.374.977/0001-93", 
    "15.385.196/0001-57", "07.763.474/0001-34", "22.266.175/0013-11", 
    "00.395.792/0001-40", "09.305.290/0001-56", "92.872.100/0001-26", 
    "01.518.211/0001-83", "04.727.154/0001-30", "17.479.056/0001-73", 
    "09.136.347/0001-30", "15.148.505/0001-75", "33.050.196/0001-88", 
    "02.016.439/0001-38", "00.592.526/0001-08", "42.513.101/0001-06", 
    "33.660.564/0001-00", "54.640.990/0001-51", "01.461.265/0001-50", 
    "51.597.300/0001-30", "00.086.413/0001-30", "43.638.022/0001-89", 
    "33.485.541/0001-08", "17.157.777/0001-67", "60.942.638/0001-73", 
    "60.736.279/0001-06", "33.042.953/0001-71", "63.899.157/0001-10", 
    "69.317.337/0001-23", "49.032.337/0001-70"
]

print(f"Total de documentos a processar: {len(document_list)}")

Total de documentos a processar: 68


In [ ]:
def fetch_company_data(document):
    """
    Realiza as chamadas em ordem e consolida os dados usando a x-api-key.
    """
    body = {
        "identifier": 1,
        "document": document,
        "webhook_url": "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40",
        "async": False
    }
    try:
        print("Consultando /registration...")
        reg_response = requests.post(ENDPOINTS["registration"], headers=HEADERS, json=body)
        reg_response.raise_for_status()
        reg_payload = reg_response.json()

        print("Consultando /business-participation...")
        part_response = requests.post(ENDPOINTS["participation"], headers=HEADERS, json=body)
        part_response.raise_for_status()
        part_payload = part_response.json()

        return reg_payload, part_payload

    except requests.exceptions.RequestException as e:
        print(f"Erro na chamada da API: {e}")
        return None, None

def process_data(reg_payload, part_payload):
    """
    Processa os dados retornando dois DataFrames: um para o resumo e outro para os sócios ativos.
    """
    # Acesso aos dados base
    reg_data = reg_payload.get("data", [{}])[0]
    part_data = part_payload.get("data", [{}])[0].get("response", {})
    doc_inicial = reg_data.get("document", "N/A")

    # 1. Processamento de Telefones e Emails
    telefones_raw = reg_data.get("phones", [])
    telefones = [str(t.get("phone")) for t in telefones_raw if t.get("phone")]
    
    emails_raw = reg_data.get("emails", [])
    emails = [str(e.get("email")) for e in emails_raw if e.get("email")]

    # 2. Processamento Detalhado de Sócios
    socios_raw = part_data.get("partners", [])
    lista_socios_detalhada = []
    nomes_socios_resumo = []
    
    for socio in socios_raw:
        status_socio = str(socio.get("status", "")).strip().lower()
        
        if status_socio not in ["inativo", "inativa"]:
            nome = socio.get("name", "Desconhecido")
            doc_socio = socio.get("document", "N/A")
            
            # Para a tabela detalhada
            lista_socios_detalhada.append({
                "Documento Empresa": doc_inicial,
                "Documento do Sócio": doc_socio,
                "Nome do Sócio": nome
            })
            # Para manter o nome no resumo (opcional)
            nomes_socios_resumo.append(nome)

    # DataFrame de Resumo (1 linha)
    resumo = {
        "Documento": doc_inicial,
        "Razão Social": reg_data.get("name", "N/A"),
        "Nome Fantasia": reg_data.get("fantasy_name", "N/A"),
        "Telefones": ", ".join(telefones) if telefones else "N/A",
        "Emails": ", ".join(emails) if emails else "N/A",
        "CNAE": reg_data.get("economic_activity", "N/A"),
        "Status da Empresa": reg_data.get("fiscal_situation", "N/A"),
        "Qtd Sócios Ativos": len(lista_socios_detalhada)
    }
    
    df_resumo = pd.DataFrame([resumo])
    df_socios = pd.DataFrame(lista_socios_detalhada)
    
    return df_resumo, df_socios

print("Funções de processamento atualizadas com a estrutura real da API.")

Funções de processamento atualizadas com a estrutura real da API.


In [11]:
# Bloco 4 - Execução Consolidada com Múltiplas Abas (Incluindo Erros)
import pandas as pd

todos_resumos = []
todos_socios = []
documentos_com_erro = []

# 1. Fase de Extração
for doc in document_list:
    print(f"Processando documento: {doc}")
    reg, part = fetch_company_data(doc)

    if reg and part:
        df_res, df_soc = process_data(reg, part)
        todos_resumos.append(df_res)
        todos_socios.append(df_soc)
    else:
        # Se entrar aqui, o documento deu erro ou é inválido para a API
        print(f"  -> Falha ao extrair dados de: {doc}")
        documentos_com_erro.append(doc)

print("\n--- Gerando Arquivo Final ---")

# 2. Consolidação das tabelas
df_final_empresas = pd.concat(todos_resumos, ignore_index=True) if todos_resumos else pd.DataFrame()
df_final_socios = pd.concat(todos_socios, ignore_index=True) if todos_socios else pd.DataFrame()
df_final_erros = pd.DataFrame(documentos_com_erro, columns=["Documento"])

# 3. Gravação no Excel com múltiplas abas
if not df_final_empresas.empty or not df_final_erros.empty:
    file_name = "relatorio_consolidado.xlsx"
    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        # Aba de Empresas com sucesso
        if not df_final_empresas.empty:
            df_final_empresas.to_excel(writer, sheet_name='Empresas', index=False)
        
        # Aba de Sócios ativos
        if not df_final_socios.empty:
            df_final_socios.to_excel(writer, sheet_name='Sócios Ativos', index=False)
            
        # Aba de Documentos Inválidos ou com Erro
        if not df_final_erros.empty:
            df_final_erros.to_excel(writer, sheet_name='Erros ou Inválidos', index=False)

    print(f"Sucesso! Arquivo '{file_name}' criado com as abas: 'Empresas', 'Sócios Ativos' e 'Erros ou Inválidos'.")
    
    # Exibe prévias no notebook
    if not df_final_empresas.empty:
        print("\nEmpresas Processadas:")
        display(df_final_empresas.head())
    
    if not df_final_erros.empty:
        print("\nDocumentos com Erro/Inválidos:")
        display(df_final_erros)
else:
    print("Nenhum dado foi processado e nenhum erro foi registrado.")

Processando documento: 10.526.937/0005-82
Consultando /registration...
Consultando /business-participation...
Processando documento: 44.069.771/0001-00
Consultando /registration...
Consultando /business-participation...
Processando documento: 17.159.005/0001-64
Consultando /registration...
Consultando /business-participation...
Processando documento: 72.908.817/0001-73
Consultando /registration...
Consultando /business-participation...
Processando documento: 48.066.047/0001-84
Consultando /registration...
Consultando /business-participation...
Processando documento: 01.488.278/0001-12
Consultando /registration...
Consultando /business-participation...
Processando documento: 13.241.169/0001-85
Consultando /registration...
Consultando /business-participation...
Processando documento: 05.885.911/0001-67
Consultando /registration...
Consultando /business-participation...
Processando documento: 38.518.783/0001-72
Consultando /registration...
Consultando /business-participation...
Processand

,Documento,Razão Social,Nome Fantasia,Telefones,Emails,CNAE,Status da Empresa,Qtd Sócios Ativos
0,10526937000582,TECELAGEM PARAHYBA DO NORDESTE SA,TECELAGEM PARAHYBA DO NORDESTE SA,N/A,N/A,"1340501 - ESTAMPARIA E TEXTURIZACAO EM FIOS, T...",Inapta,0
1,44069771000100,SCANBRASIL DESPACHOS E TRANSPORTES LTDA,SCANBRASIL DESPACHOS E TRANSPORTES LTDA,N/A,N/A,0 - undefined,Baixada,0
2,17159005000164,FIACAO E TECELAGEM SAO JOSE S/A - EM RECUPERAC...,FIACAO E TECELAGEM SAO JOSE S/A - EM RECUPERAC...,"(32) 3332-1288, (32) 93332-1288, (31) 3327-669...","oscarmagalhaes398@gmail.com, oscarferreira2@gm...","1340501 - ESTAMPARIA E TEXTURIZACAO EM FIOS, T...",Ativa,4
3,72908817000173,BOSCH REXROTH LTDA,BOSCH REXROTH LTDA,"(11) 98126-1716, (51) 98136-8264, (11) 98536-6...","edson.duwe@boschrexroth.com.br, manfred2608@ho...",4663000 - COMERCIO ATACADISTA DE MAQUINAS E EQ...,Ativa,6
4,48066047000184,IMPRENSA OFICIAL DO ESTADO S/A - IMESP,IMPRENSA OFICIAL DO ESTADO S/A - IMESP,"(11) 2799-9800, (11) 2799-9873, (11) 98906-233...","jboliveira@jbo.com.br, ilmonteiro@terra.com.br...",5822101 - EDICAO INTEGRADA A IMPRESSAO DE JORN...,Baixada,0



Documentos com Erro/Inválidos:


,Documento
0,43.638.022/0001-89
1,33.485.541/0001-08
